# Lab04 - SMEM

In [40]:
import numpy as np
import numba
import math
import time
from numba import cuda

cuda.detect()

Found 1 CUDA devices
id 0    b'NVIDIA GeForce RTX 2060'                              [SUPPORTED]
                      Compute Capability: 7.5
                           PCI Device ID: 0
                              PCI Bus ID: 1
                                    UUID: GPU-006291b6-94bd-bac8-588d-4a97f26284e1
                                Watchdog: Enabled
                            Compute Mode: WDDM
             FP32/FP64 Performance Ratio: 32
Summary:
	1/1 devices are supported


True

## Ex. 1) Parallel Reduction w/ SMEM

In [ ]:
# Without SMEM:

SIZE = 1024 * 1024

BLOCK_SIZE = min(SIZE, 1024)
GRID_SIZE = (SIZE + BLOCK_SIZE - 1) // BLOCK_SIZE

print(f"{GRID_SIZE} Blocks of {BLOCK_SIZE} Threads each")

arr_host = np.random.uniform(0, 100, SIZE).astype(np.float32)
#arr_host = np.ones(SIZE)
out_host = np.zeros(GRID_SIZE) # One per Block

arr_device = cuda.to_device(arr_host)
out_device = cuda.device_array_like(out_host)

print(arr_host)

@cuda.jit
def reduce(arr, out):
    global_idx = cuda.grid(1)
    thread_idx = cuda.threadIdx.x
    block_idx = cuda.blockIdx.x
    first_thread_in_block_index = block_idx * cuda.blockDim.x
    
    if global_idx >= len(arr):
        return
    
    
    stride = cuda.blockDim.x // 2
    while stride > 0:
        if thread_idx < stride:
            target_idx = first_thread_in_block_index + thread_idx + stride
            arr[first_thread_in_block_index + thread_idx] += arr[target_idx]
        cuda.syncthreads()
        stride //= 2
    
    if thread_idx == 0:
        out[block_idx] = arr[first_thread_in_block_index]
        
        
reduce[GRID_SIZE, BLOCK_SIZE](arr_device, out_device)
cuda.synchronize()

out_host = out_device.copy_to_host()

expected = np.sum(arr_host)
result = np.sum(out_host)

print(f"Expected: {expected}")
print(f"Got: {result}")
assert math.isclose(expected, result, rel_tol=1e-5)

1022 Blocks of 1024 Threads each
[45.415886  66.923836  22.819008  ... 61.494495   3.0286505 10.33391  ]
Expected: 52276800.0
Got: 52276798.58131027


In [103]:
# With SMEM:

SIZE = 1024 * 1024 * 1024

BLOCK_SIZE = min(SIZE, 1024)
SMEM_SIZE = BLOCK_SIZE
GRID_SIZE = (SIZE + BLOCK_SIZE - 1) // BLOCK_SIZE

print(f"{GRID_SIZE} Blocks of {BLOCK_SIZE} Threads each")

arr_host = np.random.randint(0, 100, SIZE)
#arr_host = np.random.uniform(0, 100, SIZE).astype(np.float32)
#arr_host = np.ones(SIZE)
out_host = np.zeros(GRID_SIZE) # One per Block

arr_device = cuda.to_device(arr_host)
out_device = cuda.device_array_like(out_host)

print(arr_host)

@cuda.jit
def reduce(arr, out):
    global_idx = cuda.grid(1)
    thread_idx = cuda.threadIdx.x
    block_idx = cuda.blockIdx.x    
    
    # 1st: all threads contribute to copying the portion of GMEM into SMEM
    smem = cuda.shared.array(SMEM_SIZE, dtype=np.float32)
    if global_idx < len(arr):
        smem[thread_idx] = arr[global_idx]
    else:
        smem[thread_idx] = 0.0
    cuda.syncthreads()
    
    # 2nd: threads go on with the computation (using smem instead of arr!)
    stride = cuda.blockDim.x // 2
    while stride > 0:
        if thread_idx < stride:
            target_idx = thread_idx + stride
            smem[thread_idx] += smem[target_idx]
        cuda.syncthreads()
        stride //= 2
    
    if thread_idx == 0:
        out[block_idx] = smem[0]
        
        
reduce[GRID_SIZE, BLOCK_SIZE](arr_device, out_device)
cuda.synchronize()

out_host = out_device.copy_to_host()

expected = np.sum(arr_host)
result = np.sum(out_host)

print(f"Expected: {expected}")
print(f"Got: {result}")
assert math.isclose(expected, result, rel_tol=1e-5)

1048576 Blocks of 1024 Threads each
[ 4 72 25 ... 94 73 90]
Expected: 53149190981
Got: 53149190981.0


## Ex. 2) Matrix Multiplication w/ SMEM

In [130]:
# Without SMEM:

SIZE_A = (1000, 3252)
SIZE_B = (3252, 315)
SIZE_C = (SIZE_A[0], SIZE_B[1])

BLOCK_SIZE = (32, 32)
GRID_SIZE = ((SIZE_C[1] + BLOCK_SIZE[0] - 1) // BLOCK_SIZE[0], (SIZE_C[0] + BLOCK_SIZE[1] - 1) // BLOCK_SIZE[1] )

print(f"Grid Size: {GRID_SIZE}")
print(f"Block Size: {BLOCK_SIZE}")

A_host = np.random.randint(0, 10, SIZE_A)
B_host = np.random.randint(0, 10, SIZE_B)
C_host = np.zeros(SIZE_C)

A_device = cuda.to_device(A_host)
B_device = cuda.to_device(B_host)
C_device = cuda.device_array_like(C_host)

@cuda.jit
def mat_mul(A, B, C):
    col, row = cuda.grid(2)
    
    if row >= C.shape[0] or col >= C.shape[1]:
        return
    
    tmp = 0.0
    for i in range(A.shape[1]):
        tmp += A[row, i] * B[i, col]
    C[row, col] = tmp
   
mat_mul[GRID_SIZE, BLOCK_SIZE](A_device, B_device, C_device)
cuda.synchronize()
 
expected = A_host @ B_host

result = C_device.copy_to_host()

print(f"Expected: {expected}")
print(f"Got: {result}")

Grid Size: (10, 32)
Block Size: (32, 32)
Expected: [[65270 65978 65607 ... 64715 64800 65122]
 [66315 65461 66155 ... 65406 65334 64610]
 [65363 66158 65824 ... 65221 65173 65389]
 ...
 [67045 66536 67195 ... 66525 67053 67844]
 [65524 66401 66423 ... 65335 64452 65992]
 [66049 65335 65980 ... 65636 66225 65557]]
Got: [[65270. 65978. 65607. ... 64715. 64800. 65122.]
 [66315. 65461. 66155. ... 65406. 65334. 64610.]
 [65363. 66158. 65824. ... 65221. 65173. 65389.]
 ...
 [67045. 66536. 67195. ... 66525. 67053. 67844.]
 [65524. 66401. 66423. ... 65335. 64452. 65992.]
 [66049. 65335. 65980. ... 65636. 66225. 65557.]]


In [155]:
# With SMEM:

SIZE_A = (1027, 2047)
SIZE_B = (2047, 1248)
SIZE_C = (SIZE_A[0], SIZE_B[1])

BLOCK_SIZE = (32, 32)
GRID_SIZE = ((SIZE_C[1] + BLOCK_SIZE[0] - 1) // BLOCK_SIZE[0], (SIZE_C[0] + BLOCK_SIZE[1] - 1) // BLOCK_SIZE[1] )

print(f"Grid Size: {GRID_SIZE}")
print(f"Block Size: {BLOCK_SIZE}")

A_host = np.random.randint(0, 10, SIZE_A)
B_host = np.random.randint(0, 10, SIZE_B)
C_host = np.zeros(SIZE_C)

A_device = cuda.to_device(A_host)
B_device = cuda.to_device(B_host)
C_device = cuda.device_array_like(C_host)

@cuda.jit
def mat_mul(A, B, C):
    global_idx_col, global_idx_row = cuda.grid(2)
    thread_idx_col, thread_idx_row = cuda.threadIdx.x, cuda.threadIdx.y
    
    # Notice that we remove the early return from here, cause those threads should still 
    # contribute copying into the SMEM, not return instantly!
    
    # Each block allocates room for a tile from A and a tile from B
    # The tiles are of the size of the block, as we want 1 thread <-> 1 value in C
    smem_A = cuda.shared.array(BLOCK_SIZE, dtype=numba.float32)
    smem_B = cuda.shared.array(BLOCK_SIZE, dtype=numba.float32)
    
    # Now we want to copy different tiles into smem_A and smem_B
    tiles_count = (A.shape[1] + BLOCK_SIZE[0] - 1) // BLOCK_SIZE[0] 
    acc = 0.0
    for tile in range(tiles_count):
        col_A = BLOCK_SIZE[0] * tile + thread_idx_col
        row_B = BLOCK_SIZE[0] * tile + thread_idx_row
        
        # 1st: each thread copies one value from A and one value from B
        # this copies the two tiles from A and B into smem_A and smem_B
        if global_idx_row < A.shape[0] and col_A < A.shape[1]:
            smem_A[thread_idx_row, thread_idx_col] = A[global_idx_row, col_A]
        else:
            smem_A[thread_idx_row, thread_idx_col] = 0.0
            
        if row_B < B.shape[0] and global_idx_col < B.shape[1]:
            smem_B[thread_idx_row, thread_idx_col] = B[row_B, global_idx_col]
        else:
            smem_B[thread_idx_row, thread_idx_col] = 0.0

        cuda.syncthreads()
        
        # 2nd: it accumulates into acc the partial results from the two tiles  
        for i in range(BLOCK_SIZE[0]):
            acc += smem_A[thread_idx_row, i] * smem_B[i, thread_idx_col]
        
        cuda.syncthreads()
    
    if global_idx_row < C.shape[0] and global_idx_col < C.shape[1]:
        C[global_idx_row, global_idx_col] = acc
   
mat_mul[GRID_SIZE, BLOCK_SIZE](A_device, B_device, C_device)
cuda.synchronize()
 
expected = A_host @ B_host

result = C_device.copy_to_host()

print(f"Expected: {expected}")
print(f"Got: {result}")

Grid Size: (39, 33)
Block Size: (32, 32)
Expected: [[39699 41594 41291 ... 40261 40600 40415]
 [39507 41085 41438 ... 39647 40593 40513]
 [39931 42381 41558 ... 41631 41238 41232]
 ...
 [40227 42839 42335 ... 40902 41894 41972]
 [39874 42713 42202 ... 40816 41876 41372]
 [40989 42045 42660 ... 41348 41695 41826]]
Got: [[39699. 41594. 41291. ... 40261. 40600. 40415.]
 [39507. 41085. 41438. ... 39647. 40593. 40513.]
 [39931. 42381. 41558. ... 41631. 41238. 41232.]
 ...
 [40227. 42839. 42335. ... 40902. 41894. 41972.]
 [39874. 42713. 42202. ... 40816. 41876. 41372.]
 [40989. 42045. 42660. ... 41348. 41695. 41826.]]


## Ex. 3) Convolution w/ SMEM

In [253]:
# Without SMEM:

SIZE = 1024
MASK_RADIUS = 3
MASK_SIZE = 2 * MASK_RADIUS + 1

BLOCK_SIZE = min(32, SIZE)
GRID_SIZE = (SIZE + BLOCK_SIZE - 1) // BLOCK_SIZE

arr_host = np.random.randint(0, 10, SIZE)
mask_host = np.random.randint(0, 10, MASK_SIZE)
out_host = np.zeros(SIZE)

print(arr_host)
print(mask_host)

arr_device = cuda.to_device(arr_host)
mask_device = cuda.to_device(mask_host)
out_device = cuda.device_array(SIZE)

@cuda.jit
def conv_1d(arr, mask, out):
    idx = cuda.grid(1)
    
    if idx >= len(arr):
        return
    
    mask_start_idx = idx - MASK_RADIUS
    mask_end_idx = idx + MASK_RADIUS
    
    acc = 0.0
    mask_idx = len(mask) - 1
    for j in range(mask_start_idx, mask_end_idx + 1):
        if j >= 0 and j < len(arr):
            acc += arr[j] * mask[mask_idx]
        mask_idx -= 1

    out[idx] = acc
    
expected = np.convolve(arr_host, mask_host, mode='same')

conv_1d[GRID_SIZE, BLOCK_SIZE](arr_device, mask_device, out_device)
cuda.synchronize()

out_host = out_device.copy_to_host()

print(f"Expected: {expected}")
print(f"Got: {out_host}")

assert all([a == b for a, b in zip(expected, out_host)])


[9 0 2 ... 3 9 7]
[2 7 2 7 3 0 1]
Expected: [ 79  54  76 ... 106  92  81]
Got: [ 79.  54.  76. ... 106.  92.  81.]


c:\Users\Filippo Corti\Documents\GitHub\GPUComputing\.venv\Lib\site-packages\numba\cuda\dispatcher.py:536: NumbaPerformanceWarning: Grid size 32 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


In [296]:
# With SMEM:

# Basically we need to load into SMEM whatever the block needs from arr to compute out
# That is a vector of size BLOCK_SIZE + MASK_SIZE - 1

SIZE = 8
MASK_RADIUS = 1
MASK_SIZE = 2 * MASK_RADIUS + 1

BLOCK_SIZE = min(32, SIZE)
GRID_SIZE = (SIZE + BLOCK_SIZE - 1) // BLOCK_SIZE

SMEM_SIZE = BLOCK_SIZE + MASK_SIZE - 1

arr_host = np.random.randint(0, 10, SIZE)
mask_host = np.random.randint(0, 10, MASK_SIZE)
out_host = np.zeros(SIZE)

print(arr_host)
print(mask_host)

arr_device = cuda.to_device(arr_host)
mask_device = cuda.to_device(mask_host)
out_device = cuda.device_array_like(out_host)

@cuda.jit
def conv_1d(arr, mask, out):
    global_idx = cuda.grid(1)
    thread_idx = cuda.threadIdx.x
    
    # 1st: load data into the SMEM
    smem = cuda.shared.array(SMEM_SIZE, dtype=numba.float32)
    
    # left halo: the first MASK_RADIUS threads copy a piece in here
    if thread_idx < MASK_RADIUS:
        if global_idx - MASK_RADIUS >= 0:
            smem[thread_idx] = arr[global_idx - MASK_RADIUS]
        else:
            smem[thread_idx] = 0.0
    
    # easy part: each thread copies one cell in the middle
    if global_idx < len(arr):
        smem[MASK_RADIUS + thread_idx] = arr[global_idx]
    else:
        smem[MASK_RADIUS + thread_idx] = 0.0
    
    # right halo: the first MASK_RADIUS threads copy a piece in here
    if thread_idx < MASK_RADIUS:
        if global_idx + BLOCK_SIZE < len(arr):
            smem[BLOCK_SIZE + MASK_RADIUS + thread_idx] = arr[global_idx + BLOCK_SIZE]
        else:
            smem[BLOCK_SIZE + MASK_RADIUS + thread_idx] = 0.0
        
    cuda.syncthreads()
    
    if global_idx >= len(arr):
        return
    
    mask_start_idx = thread_idx
    mask_end_idx = thread_idx + MASK_SIZE - 1
    
    acc = 0.0
    mask_idx = MASK_SIZE - 1
    for j in range(mask_start_idx, mask_end_idx + 1):
        acc += smem[j] * mask[mask_idx]
        mask_idx -= 1

    out[global_idx] = acc
    
expected = np.convolve(arr_host, mask_host, mode='same')

conv_1d[GRID_SIZE, BLOCK_SIZE](arr_device, mask_device, out_device)
cuda.synchronize()

out_host = out_device.copy_to_host()

print(f"Expected: {expected}")
print(f"Got: {out_host}")

assert all([a == b for a, b in zip(expected, out_host)])


[5 1 4 0 7 8 8 6]
[1 5 4]
Expected: [26 29 24 23 43 76 78 62]
Got: [26. 29. 24. 23. 43. 76. 78. 62.]


c:\Users\Filippo Corti\Documents\GitHub\GPUComputing\.venv\Lib\site-packages\numba\cuda\dispatcher.py:536: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
